# Clustering and Association Rules

The final unsupervised ideas: grouping similar customers with hierarchical and K-means clustering, and finding items that occur together with the Apriori algorithm.

## Imports and loading helper

In [ ]:
# Standard imports for the whole course
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Find the Datasets folder whether the notebook is run from the
# project root or from inside the notebooks folder.
from pathlib import Path
for candidate in [Path("Datasets"), Path("../Datasets")]:
    if candidate.exists():
        DATA = candidate
        break

def load(name):
    """Load a CSV from the Datasets folder by file name."""
    return pd.read_csv(DATA / name)

print("Datasets are in:", DATA.resolve())
print(sorted(p.name for p in DATA.glob("*.csv")))

## Preparing the mall customer data

The file has a few missing rows, which we drop before clustering.

In [ ]:
df = load("Mall_Customers_Clustering.csv")
df = df.dropna()
X = df.iloc[:, [3, 4]].values   # annual income and spending score
print(df.head())

## Hierarchical clustering

The dendrogram records every merge. The height at which we cut it decides the number of clusters.

In [ ]:
import scipy.cluster.hierarchy as shc
plt.figure(figsize=(9, 4))
shc.dendrogram(shc.linkage(X, method="ward"))
plt.title("Dendrogram for mall customers")
plt.xlabel("Customers"); plt.ylabel("Euclidean distance"); plt.show()

## Agglomerative clustering with five clusters

In [ ]:
from sklearn.cluster import AgglomerativeClustering
labels = AgglomerativeClustering(n_clusters=5, linkage="ward").fit_predict(X)
print(np.bincount(labels))

## K-means and the elbow method

We fit K-means for one to ten clusters and look for the bend in the within cluster sum of squares.

In [ ]:
from sklearn.cluster import KMeans
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
    km.fit(X)
    wcss.append(km.inertia_)

plt.plot(range(1, 11), wcss, "o-")
plt.xlabel("Number of clusters"); plt.ylabel("WCSS")
plt.title("The elbow method"); plt.show()

## Five clusters and their centres

In [ ]:
km = KMeans(n_clusters=5, init="k-means++", random_state=42, n_init=10)
labels = km.fit_predict(X)

plt.figure(figsize=(7, 4.5))
for i, colour in enumerate(["tab:blue", "tab:orange", "tab:green",
                            "tab:red", "tab:purple"]):
    mask = labels == i
    plt.scatter(X[mask, 0], X[mask, 1], s=22, color=colour,
                label=f"cluster {i + 1}")
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            marker="X", s=180, color="black", label="centroids")
plt.xlabel("Annual income (k$)"); plt.ylabel("Spending score")
plt.legend(); plt.show()

## Association rules with Apriori

Apriori finds items that occur together in transactions. Here we generate a small synthetic basket log and mine it. Swap in a real transaction file to use it in practice.

In [ ]:
try:
    from apyori import apriori
    records = [["bread", "butter"],
               ["bread", "butter", "jam"],
               ["bread", "butter"],
               ["bread", "butter"],
               ["bread", "butter", "jam"],
               ["milk"]]
    rules = list(apriori(records, min_support=0.3,
                         min_confidence=0.6, min_lift=1.1, min_length=2))
    for rule in rules:
        print("Rule:", list(rule.items))
        print("  support   :", round(rule.support, 3))
        for stat in rule.ordered_statistics:
            print("  ", list(stat.items_base), "->", list(stat.items_add),
                  "conf", round(stat.confidence, 3),
                  "lift", round(stat.lift, 3))
except ImportError:
    print("apyori is not installed; run pip install apyori")

That completes the course notebooks. Try changing a parameter, such as the number of clusters or the polynomial degree, and watch how the results move.